# MSI Pipeline - Script 06: Differential Analysis

This notebook performs differential analysis comparing groups of samples or clusters.

## Features
- Group definition (e.g., tumor vs normal, aging vs normal)
- Per-group variance calculation
- Peak classification:
  - Group-specific peaks (variable in one group only)
  - Shared peaks (variable in both)
- Clustermap visualization
- Cross-sample consistency filtering

## Input
- Clustered AnnData files from script05
- Group labels (sample metadata)

## Output
- Differential peak lists (CSV)
- Clustermap visualizations (PNG)
- Variance analysis results

In [ ]:
import sys
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

from utils import io as msi_io

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Plot settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

## Configuration

In [ ]:
# === CONFIGURE THESE PATHS ===

# Base data directory
BASE_DIR = Path(r"T:/Sammy Data/Third set results/")  # Path.home() / "ext_hd_sammy"
#BASE_DIR = Path.home() / "ext_hd_sammy"

# Input: Clustered AnnData files
INPUT_DIR = BASE_DIR / "peptides_h5ad_processed"

# Output directory
OUTPUT_DIR = BASE_DIR / "peptides_differential_mla_endo"

# Comparison name (for output files)
COMPARISON_NAME = "MLA_vs_Endo"

# Group definitions (sample_id -> group)
# Modify this based on your experimental design
SAMPLE_GROUPS = {
    # Example:
     'S025 ovary manual peptides': 'MLA',
     'S033 ovary manual peptides': 'MLA',
     'S034 ovary manual peptides': 'MLA',
     'S045 ovary manual peptides': 'MLA',
     'S047 ovary manual peptides': 'MLA',
     'SO48 ovary manual peptides': 'MLA',
     'SO50 ovary manual peptides': 'MLA',
     'SO51 ovary manual peptides': 'MLA',  # Note: missing space after SO51 in filename
     'S024 ovary manual peptides': 'Endo',
     'S030 ovary manual peptides': 'Endo',
     'S032 ovary manual peptides': 'Endo',
     'S044 ovary manual peptides': 'Endo',
     'S046 ovary manual peptides': 'Endo',
     'SO58 ovary manual peptides': 'Endo',
     'SO59 ovary manual peptides': 'Endo'
}

# Variance thresholds
VARIANCE_PERCENTILE = 75  # Consider channels above this percentile as "variable"
MIN_SAMPLES_PER_GROUP = 2  # Minimum samples to consider a group

# Create output directory
comparison_dir = OUTPUT_DIR / COMPARISON_NAME
comparison_dir.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {comparison_dir}")
print(f"Comparison: {COMPARISON_NAME}")

## Load Samples

In [ ]:
# Find all h5ad files
sample_files = list(INPUT_DIR.glob("*.h5ad"))
sample_ids = [f.stem for f in sample_files]

print(f"Found {len(sample_ids)} samples:")
for sid in sample_ids:
    group = SAMPLE_GROUPS.get(sid, 'unknown')
    print(f"  - {sid}: {group}")

In [ ]:
# Load all samples into a dictionary
adatas = {}
failed_files = []

for sample_file in sample_files:
    sample_id = sample_file.stem
    print(f"Loading {sample_id}...")
    try:
        adata = ad.read_h5ad(sample_file)
        
        # Add sample_id to obs if not present
        if 'sample_id' not in adata.obs.columns:
            adata.obs['sample_id'] = sample_id
        
        # Add group assignment
        adata.obs['group'] = SAMPLE_GROUPS.get(sample_id, 'unknown')
        
        adatas[sample_id] = adata
    except Exception as e:
        print(f"  ❌ ERROR loading {sample_id}: {e}")
        failed_files.append(sample_id)
        continue

print(f"\nSuccessfully loaded {len(adatas)} samples")
if failed_files:
    print(f"Failed to load {len(failed_files)} samples: {failed_files}")
    print("\nTip: Try closing any programs that might have these files open,")
    print("or check if the files are corrupted.")

## Compute Per-Sample Variance

In [ ]:
def compute_sample_variance(adata):
    """Compute variance per channel for a sample."""
    X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    return np.var(X, axis=0)


def compute_sample_mean(adata):
    """Compute mean per channel for a sample."""
    X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    return np.mean(X, axis=0)

In [ ]:
# Compute variance for each sample
variance_data = {}
mean_data = {}

# Get common channels across all samples
all_channels = [set(adata.var_names) for adata in adatas.values()]
common_channels = list(set.intersection(*all_channels))
print(f"Common channels across samples: {len(common_channels)}")

for sample_id, adata in adatas.items():
    # Subset to common channels
    adata_common = adata[:, common_channels].copy()
    
    variance_data[sample_id] = compute_sample_variance(adata_common)
    mean_data[sample_id] = compute_sample_mean(adata_common)

# Create DataFrames
variance_df = pd.DataFrame(variance_data, index=common_channels)
mean_df = pd.DataFrame(mean_data, index=common_channels)

print(f"\nVariance matrix shape: {variance_df.shape}")
variance_df.head()

## Group Analysis

In [ ]:
# Get samples per group
groups = {}
for sample_id in adatas.keys():
    group = SAMPLE_GROUPS.get(sample_id, 'unknown')
    if group not in groups:
        groups[group] = []
    groups[group].append(sample_id)

print("Groups:")
for group, samples in groups.items():
    print(f"  {group}: {samples}")

In [ ]:
SAMPLE_GROUPS

In [ ]:
def compute_group_stats(df, group_samples):
    """Compute mean and std across samples in a group."""
    group_df = df[group_samples]
    return {
        'mean': group_df.mean(axis=1),
        'std': group_df.std(axis=1),
        'median': group_df.median(axis=1),
    }


# Compute group-level statistics
group_variance_stats = {}
group_mean_stats = {}

for group, samples in groups.items():
    if len(samples) >= MIN_SAMPLES_PER_GROUP:
        group_variance_stats[group] = compute_group_stats(variance_df, samples)
        group_mean_stats[group] = compute_group_stats(mean_df, samples)
        print(f"Computed stats for {group} ({len(samples)} samples)")
    else:
        print(f"Skipping {group} ({len(samples)} < {MIN_SAMPLES_PER_GROUP} samples)")

## Identify Variable Peaks

In [ ]:
def identify_variable_peaks(variance_series, percentile=75):
    """Identify peaks above variance percentile threshold."""
    threshold = np.percentile(variance_series, percentile)
    return variance_series[variance_series >= threshold].index.tolist()


# Identify variable peaks per group
variable_peaks = {}

for group, group_stats in group_variance_stats.items():
    peaks = identify_variable_peaks(group_stats['mean'], VARIANCE_PERCENTILE)
    variable_peaks[group] = set(peaks)
    print(f"{group}: {len(peaks)} variable peaks")

In [ ]:
# Classify peaks
if len(variable_peaks) >= 2:
    group_names = list(variable_peaks.keys())
    
    # Get all variable peaks
    all_variable = set.union(*variable_peaks.values())
    
    # Shared peaks (variable in all groups)
    shared_peaks = set.intersection(*variable_peaks.values())
    
    # Group-specific peaks
    group_specific = {}
    for group, peaks in variable_peaks.items():
        other_peaks = set.union(*[p for g, p in variable_peaks.items() if g != group])
        group_specific[group] = peaks - other_peaks
    
    print(f"\nPeak Classification:")
    print(f"  Total variable peaks: {len(all_variable)}")
    print(f"  Shared (variable in all groups): {len(shared_peaks)}")
    for group, peaks in group_specific.items():
        print(f"  {group}-specific: {len(peaks)}")
else:
    print("Need at least 2 groups for comparison")

## Statistical Testing (Between Groups)

In [ ]:
def differential_test(mean_df, variance_df, group1_samples, group2_samples):
    """Perform differential testing between two groups."""
    results = []
    
    for channel in mean_df.index:
        # Get values for each group
        vals1 = mean_df.loc[channel, group1_samples].values
        vals2 = mean_df.loc[channel, group2_samples].values
        
        # Skip if too few samples
        if len(vals1) < 2 or len(vals2) < 2:
            continue
        
        # Skip if all values are identical (zero variance)
        if np.std(vals1) == 0 and np.std(vals2) == 0:
            t_stat, p_value = np.nan, 1.0
        else:
            # T-test
            try:
                t_stat, p_value = ttest_ind(vals1, vals2)
                # Handle NaN from ttest (can happen with zero variance in one group)
                if np.isnan(p_value):
                    p_value = 1.0
            except Exception as e:
                print(f"Warning: t-test failed for {channel}: {e}")
                t_stat, p_value = np.nan, 1.0
        
        # Log fold change
        mean1 = np.mean(vals1)
        mean2 = np.mean(vals2)
        log_fc = np.log2((mean1 + 1) / (mean2 + 1))
        
        results.append({
            'channel': channel,
            'mean_group1': mean1,
            'mean_group2': mean2,
            'log2_fc': log_fc,
            't_statistic': t_stat,
            'p_value': p_value,
        })
    
    results_df = pd.DataFrame(results)
    
    # FDR correction
    from scipy.stats import false_discovery_control
    valid_pvals = results_df['p_value'].dropna()
    if len(valid_pvals) > 0:
        results_df['p_adjusted'] = np.nan
        results_df.loc[results_df['p_value'].notna(), 'p_adjusted'] = \
            false_discovery_control(valid_pvals.values)
    
    return results_df

In [ ]:
# Perform differential testing between groups
if len(groups) >= 2:
    # Get named groups (exclude 'unknown')
    named_groups = {k: v for k, v in groups.items() if k != 'unknown'}
    
    if len(named_groups) >= 2:
        group_names = list(named_groups.keys())
        group1, group2 = group_names[0], group_names[1]
    else:
        # Fall back to all groups if not enough named groups
        group_names = list(groups.keys())
        group1, group2 = group_names[0], group_names[1]
    
    print(f"Comparing {group1} vs {group2}...")
    print(f"  Group 1 ({group1}): {len(groups[group1])} samples")
    print(f"  Group 2 ({group2}): {len(groups[group2])} samples")
    
    diff_results = differential_test(
        mean_df, 
        variance_df,
        groups[group1], 
        groups[group2]
    )
    
    # Sort by significance
    diff_results = diff_results.sort_values('p_value')
    
    print(f"\nTop differential peaks:")
    print(diff_results.head(10))

## Visualizations

In [ ]:
# Volcano plot
if 'diff_results' in dir() and len(diff_results) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Calculate -log10(p)
    diff_results['neg_log_p'] = -np.log10(diff_results['p_value'] + 1e-300)
    
    # Color by significance
    # Check if p_adjusted exists, otherwise use p_value
    if 'p_adjusted' in diff_results.columns:
        sig_mask = (diff_results['p_adjusted'] < 0.05) & (np.abs(diff_results['log2_fc']) > 1)
    else:
        print("Warning: p_adjusted not available, using p_value instead")
        sig_mask = (diff_results['p_value'] < 0.05) & (np.abs(diff_results['log2_fc']) > 1)
    
    ax.scatter(
        diff_results.loc[~sig_mask, 'log2_fc'],
        diff_results.loc[~sig_mask, 'neg_log_p'],
        alpha=0.5, s=20, c='gray', label='Not significant'
    )
    ax.scatter(
        diff_results.loc[sig_mask, 'log2_fc'],
        diff_results.loc[sig_mask, 'neg_log_p'],
        alpha=0.7, s=30, c='red', label='Significant'
    )
    
    ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5)
    ax.axvline(-1, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(1, color='gray', linestyle='--', alpha=0.5)
    
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10(p-value)')
    ax.set_title(f'Volcano Plot: {group1} vs {group2}')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(comparison_dir / 'volcano_plot.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Clustermap of variance
if variance_df.shape[1] > 1:  # Need multiple samples
    # Select top variable channels
    top_var_channels = variance_df.mean(axis=1).nlargest(50).index
    
    # Add group colors
    sample_colors = []
    color_map = {'tumor': 'red', 'normal': 'blue', 'unknown': 'gray'}
    for sample in variance_df.columns:
        group = SAMPLE_GROUPS.get(sample, 'unknown')
        sample_colors.append(color_map.get(group, 'gray'))
    
    g = sns.clustermap(
        variance_df.loc[top_var_channels],
        cmap='viridis',
        col_colors=sample_colors,
        figsize=(12, 10),
        yticklabels=True,
        xticklabels=True,
    )
    g.ax_heatmap.set_xlabel('Sample')
    g.ax_heatmap.set_ylabel('Channel')
    plt.title('Channel Variance Across Samples')
    
    plt.savefig(comparison_dir / 'variance_clustermap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Clustermap of mean intensities for differential peaks
if 'diff_results' in dir() and len(diff_results) > 0:
    # Get significant peaks
    if 'p_adjusted' in diff_results.columns:
        sig_peaks = diff_results[diff_results['p_adjusted'] < 0.05]['channel'].tolist()[:50]
    else:
        sig_peaks = diff_results[diff_results['p_value'] < 0.05]['channel'].tolist()[:50]
    
    if len(sig_peaks) > 5:
        g = sns.clustermap(
            mean_df.loc[sig_peaks],
            cmap='RdBu_r',
            col_colors=sample_colors,
            figsize=(12, 10),
            yticklabels=True,
            xticklabels=True,
            z_score=0,  # Row-wise z-score
        )
        g.ax_heatmap.set_xlabel('Sample')
        g.ax_heatmap.set_ylabel('Channel')
        plt.title('Differential Peaks (z-scored)')
        
        plt.savefig(comparison_dir / 'differential_clustermap.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print(f"Only {len(sig_peaks)} significant peaks, skipping clustermap")

## Save Results

In [ ]:
# Save differential results
if 'diff_results' in dir():
    diff_results.to_csv(comparison_dir / 'differential_results.csv', index=False)
    print(f"Saved differential results")

# Save variance matrix
variance_df.to_csv(comparison_dir / 'variance_matrix.csv')
print(f"Saved variance matrix")

# Save mean matrix
mean_df.to_csv(comparison_dir / 'mean_matrix.csv')
print(f"Saved mean matrix")

# Save variable peaks
if 'variable_peaks' in dir():
    for group, peaks in variable_peaks.items():
        with open(comparison_dir / f'{group}_variable_peaks.txt', 'w') as f:
            f.write(f"# Variable peaks for {group}\n")
            f.write(f"# Variance percentile threshold: {VARIANCE_PERCENTILE}\n\n")
            for peak in sorted(peaks):
                f.write(f"{peak}\n")
    print(f"Saved variable peak lists")

# Save group-specific peaks
if 'group_specific' in dir():
    for group, peaks in group_specific.items():
        with open(comparison_dir / f'{group}_specific_peaks.txt', 'w') as f:
            f.write(f"# {group}-specific peaks\n\n")
            for peak in sorted(peaks):
                f.write(f"{peak}\n")
    print(f"Saved group-specific peak lists")

# Save shared peaks
if 'shared_peaks' in dir():
    with open(comparison_dir / 'shared_variable_peaks.txt', 'w') as f:
        f.write(f"# Shared variable peaks (variable in all groups)\n\n")
        for peak in sorted(shared_peaks):
            f.write(f"{peak}\n")
    print(f"Saved shared peaks")

In [ ]:
# Summary statistics
summary = {
    'comparison': COMPARISON_NAME,
    'n_samples': len(adatas),
    'n_common_channels': len(common_channels),
    'variance_percentile': VARIANCE_PERCENTILE,
}

if 'diff_results' in dir():
    if 'p_adjusted' in diff_results.columns:
        summary['n_significant'] = (diff_results['p_adjusted'] < 0.05).sum()
        summary['n_up'] = ((diff_results['p_adjusted'] < 0.05) & (diff_results['log2_fc'] > 0)).sum()
        summary['n_down'] = ((diff_results['p_adjusted'] < 0.05) & (diff_results['log2_fc'] < 0)).sum()
    else:
        summary['n_significant'] = (diff_results['p_value'] < 0.05).sum()
        summary['n_up'] = ((diff_results['p_value'] < 0.05) & (diff_results['log2_fc'] > 0)).sum()
        summary['n_down'] = ((diff_results['p_value'] < 0.05) & (diff_results['log2_fc'] < 0)).sum()

if 'shared_peaks' in dir():
    summary['n_shared_peaks'] = len(shared_peaks)

for group, peaks in variable_peaks.items():
    summary[f'n_variable_{group}'] = len(peaks)

for group, peaks in group_specific.items():
    summary[f'n_specific_{group}'] = len(peaks)

summary_df = pd.DataFrame([summary])
summary_df.to_csv(comparison_dir / 'analysis_summary.csv', index=False)

print("\nAnalysis Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")